# 08 - PR Abandonment Analysis

Survival analysis with Kaplan-Meier curves and Cox proportional hazards,
plus abandonment classification using Random Forest and XGBoost with
ROC curve evaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from lifelines import KaplanMeierFitter

from oss_pulse.analyze.abandonment import (
    build_survival_data,
    fit_abandonment_classifier,
    fit_cox_ph,
    fit_kaplan_meier,
)
from oss_pulse.visualize.comparison import plot_roc_curve, plot_survival_curves
from oss_pulse.visualize.style import PALETTE, setup_style

setup_style()

In [ ]:
# Load classified PR data
DATA_DIR = Path("../data/processed")
pr_df = pd.read_parquet(DATA_DIR / "pr_events_featured.parquet")

print(f"PR events: {len(pr_df)} rows")
print(f"Outcome distribution:")
print(pr_df["pr_outcome"].value_counts())

In [ ]:
# Build survival data
# TODO: run with real data
surv_df = build_survival_data(pr_df)
print(f"Survival data: {len(surv_df)} PRs")
print(f"Observed events: {surv_df['event'].sum()}")
print(f"Censored: {(surv_df['event'] == 0).sum()}")
surv_df.head()

In [ ]:
# Overall Kaplan-Meier survival curve
# TODO: run with real data
kmf_all = fit_kaplan_meier(surv_df)
median_survival = kmf_all.median_survival_time_
print(f"Median PR survival time: {median_survival:.1f} days")

print("\nSurvival probabilities at key timepoints:")
for days in [7, 30, 90, 180, 365]:
    prob = kmf_all.predict(days)
    print(f"  {days:>3d} days: {prob:.3f}")

In [ ]:
# Kaplan-Meier by PR size bucket
# TODO: run with real data
curves = []
for size in ["small", "medium", "large"]:
    subset = surv_df[surv_df["pr_size_bucket"] == size]
    if len(subset) > 10:
        kmf = KaplanMeierFitter()
        kmf.fit(
            subset["duration_days"], subset["event"],
            label=f"{size} PRs",
        )
        curves.append((kmf, f"{size} PRs"))

fig = plot_survival_curves(curves, title="PR Survival by Size")
fig.show()

In [ ]:
# Cox Proportional Hazards model
# TODO: run with real data
cph = fit_cox_ph(surv_df, covariates=["author_type", "pr_size_bucket"])
print("Cox PH Summary:")
cph.print_summary()

In [ ]:
# Abandonment classifier (RF vs XGBoost)
# TODO: run with real data
features = surv_df.copy()
features["is_abandoned"] = (
    (features["event"] == 0) & (features["duration_days"] > 90)
).astype(int)

# Encode categoricals for the classifier
features_encoded = pd.get_dummies(
    features[["duration_days", "author_type", "pr_size_bucket", "is_abandoned"]],
    columns=["author_type", "pr_size_bucket"],
    drop_first=True,
)

print(f"Abandonment rate: {features['is_abandoned'].mean():.1%}")

if features["is_abandoned"].sum() >= 5:
    clf_results = fit_abandonment_classifier(features_encoded, target="is_abandoned")
    print(f"\nRandom Forest AUC: {clf_results['rf_auc']:.3f}")
    print(f"XGBoost AUC:       {clf_results['xgb_auc']:.3f}")
    print(f"Best model:        {clf_results['best_model']}")
    print(f"\nTop features:")
    print(clf_results["feature_importance"].head(10))
else:
    print("Not enough abandoned PRs to train classifier.")

In [ ]:
# ROC curves for the classifiers
# TODO: run with real data
if features["is_abandoned"].sum() >= 5:
    from sklearn.model_selection import train_test_split

    feature_cols = [c for c in features_encoded.columns if c != "is_abandoned"]
    X = features_encoded[feature_cols]
    y = features_encoded["is_abandoned"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y,
    )

    rf_proba = clf_results["rf_model"].predict_proba(X_test)[:, 1]
    xgb_proba = clf_results["xgb_model"].predict_proba(X_test)[:, 1]

    fig = plot_roc_curve(
        y_test,
        {"Random Forest": rf_proba, "XGBoost": xgb_proba},
        title="Abandonment Classifier ROC",
    )
    fig.show()